# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](image.png)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [11]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2)
        }
        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3)
        }
        self.terminal_states = {
            (2, 2) : 2,
            (0, 5) : 10
        }
        self.danger_states = {
            (1, 4): -3,
            (3, 5): -10,
            (4, 1): -3
        }
        self.living_reward = -1.0
        self.gamma = 0.9
        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state != self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        if state in self.terminal_states:
            return True
        if state in self.danger_states and self.danger_states[state] == -10:
            return True
        return False

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        if not self.is_valid_state(state):
            return 0.0
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended = 0.60
            p_perp = 0.20
        else:
            p_intended = 0.90
            p_perp = 0.05

        # perpendicular directions
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perp),
            (perp2, p_perp),
        ]

        probs = {}
        for (dr, dc), p in outcomes:
            nr, nc = state[0] + dr, state[1] + dc
            next_state = (nr, nc)
            if not self.is_valid_state(next_state):
                next_state = state
            probs[next_state] = probs.get(next_state, 0.0) + p

        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [12]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 30
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [13]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(p * V[s_next] for s_next, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # initialize V
    V = {s: 0.0 for s in grid.states()}
    for s in V:
        if grid.is_terminal(s):
            V[s] = grid.get_reward(s)

    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = V.copy()
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue

            action_values = []
            for a in grid.actions:
                ev = expected_next_value(grid, s, a, V)
                action_values.append(ev)

            best_next = max(action_values) if action_values else 0.0
            new_v = grid.get_reward(s) + grid.gamma * best_next
            delta = max(delta, abs(new_v - V[s]))
            V_new[s] = new_v

        V = V_new
        if delta < threshold:
            return V, it

    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            # arbitrary action for terminal (not used by print_policy)
            policy[s] = grid.actions[0]
            continue

        best_a = None
        best_val = -float("inf")
        for a in grid.actions:
            val = expected_next_value(grid, s, a, V)
            if val > best_val:
                best_val = val
                best_a = a
        policy[s] = best_a
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [14]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for s in V:
        if grid.is_terminal(s):
            V[s] = grid.get_reward(s)

    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = V.copy()
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue

            a = policy[s]
            ev = expected_next_value(grid, s, a, V)
            new_v = grid.get_reward(s) + grid.gamma * ev
            delta = max(delta, abs(new_v - V[s]))
            V_new[s] = new_v

        V = V_new
        if delta < threshold:
            return V, it

    return V, max_iter


def policy_improvement(grid, V):
    policy = {}
    policy_stable = True
    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = grid.actions[0]
            continue

        best_a = None
        best_val = -float("inf")
        for a in grid.actions:
            val = expected_next_value(grid, s, a, V)
            if val > best_val:
                best_val = val
                best_a = a
        policy[s] = best_a

    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # Initialize with arbitrary policy (prefer RIGHT)
    policy = {s: grid.actions[3] for s in grid.states()}
    
    history = []
    
    for it in range(max_iter):
        # Policy Evaluation
        V, _ = policy_evaluation(grid, policy, threshold)
        
        # Policy Improvement
        policy_new = policy_improvement(grid, V)
        
        # Check stability
        if policy_new == policy:
            history.append(it + 1)
            return policy_new, V, history
        
        policy = policy_new
        history.append(it + 1)
    
    return policy, V, history


## Parte 4 — Visualización y comparación


In [15]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [16]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 17

Valores:
 +0.809 |  +2.206 |  +3.787 |   WALL   |  +7.608 | +10.000
 -0.306 |   WALL   |  +2.084 |  +3.782 |  +3.675 |  +7.608
 -0.995 |  +0.117 |  +2.000 |  +2.323 |   WALL   |  +5.583
 -1.698 |  -0.658 |  +0.620 |  +0.658 |  +1.626 | -10.000
 -2.661 |  -3.684 |   WALL   |  -0.487 |  +0.236 |  -1.319

Política:
 →  |  →  |  →  |  #  |  →  | +10
 →  |  #  |  →  |  ↑  |  ↑  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ↑  | -10
 ↑  |  ↑  |  #  |  ↑  |  ↑  |  ← 

=== POLICY ITERATION ===
Historia: [1, 2, 3, 4]

Valores:
 +0.809 |  +2.206 |  +3.787 |   WALL   |  +7.608 | +10.000
 -0.306 |   WALL   |  +2.084 |  +3.782 |  +3.675 |  +7.608
 -0.995 |  +0.117 |  +2.000 |  +2.323 |   WALL   |  +5.583
 -1.698 |  -0.658 |  +0.620 |  +0.658 |  +1.626 | -10.000
 -2.661 |  -3.684 |   WALL   |  -0.487 |  +0.236 |  -1.319

Política:
 →  |  →  |  →  |  #  |  →  | +10
 →  |  #  |  →  |  ↑  |  ↑  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

**1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?**

Desde START busca la entrega +10 (la política en START es (0,1), ir a la derecha hacia (0,5)).

**2. ¿Por qué una recompensa menor podría ser óptima?**

Una recompensa menor pero más próxima puede ser óptima porque el coste por paso, la incertidumbre y el descuento reducen el valor esperado de recompensas lejanas, así un objetivo cercano con menor recompensa puede dar mayor retorno neto.


**3. ¿En qué estados el piso resbaloso cambia la decisión?**


En los estados resbalosos (1,2), (2,1) y (3,3) y en sus vecinos inmediatos, la mayor probabilidad de desviación allí puede hacer que se eviten o se rodeen dichas celdas.


**4. ¿Qué papel cumple el costo por paso `-1`?**

El coste por paso (-1) penaliza trayectorias largas, incentiva rutas más cortas y evita deambular infinito, cambia el trade-off entre recompensa y distancia.


**5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?**

Porque la dinámica es dependiente del estado, hay estados resbalosos con distintas probabilidades y las colisiones con paredes/bordes devuelven al mismo estado, por tanto T(s,a,·) varía con s.



### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.


Seguirá intentando la recompensa +10, menos penalización por pasos favorece la recompensa mayor.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.


Se evitarán rutas que crucen resbalosos, si la ruta a +10 atraviesa resbalosos, podría preferirse la estación +2.


### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?


Sí, la política valora más recompensas lejanas y aumentará la tendencia a intentar llegar al +10.


### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.



Living_reward ≈ -7 (si el coste por paso es mucho más negativo que ~-7 la política desde START podría preferir ir a +2 en vez de buscar +10).
